# Phase 4 — Multi-quarter panel, model comparison, no-look-ahead validation & tuning
### Thin walkthrough notebook — the real logic lives in `src/camel_sentinel/`

**Why this notebook exists**: `02_baseline_modeling.ipynb`'s evaluation was inconclusive — one quarter's snapshot (2023-12-31) joined against the full FDIC failure history only has **15 failed banks out of 4,658**, split across train/test as few as 3-4 test positives. No algorithm comparison on that few examples means anything; a 0.00 recall there reflects too little data, not a bad model.

**What this notebook does**, in order:
1. Pools *many* quarters (2005-2023, including the 2008-2012 crisis where most real U.S. bank failures actually happened) into one training panel, each bank-quarter labeled with a proper forward-looking target — *"did this bank fail within the next 2 years?"* — instead of Phase 3's "has this bank ever failed, at any point in FDIC history?" (see `src/camel_sentinel/features/panel.py` for why the windowed label avoids look-ahead).
2. Compares several model families on that panel with **grouped** cross-validation (grouped by bank `CERT`, since the same bank appears in many consecutive quarters — a plain random split would leak a bank's data between train and test).
3. **No-look-ahead time-based validation**: the grouped CV above still shuffles quarters from every year into every fold, so a model could in principle be doing well partly *because* it saw some 2008-2009-crisis-era pattern during training and gets tested on adjacent-year data from the same regime. The blueprint's Phase 4 explicitly asks for validation "cross-validated on time (no look-ahead)" — so this notebook adds a stricter check: train only on 2005-2011, evaluate only on 2012-2023, a period the model never saw during training or tuning at all.
4. **Hyperparameter tuning**, restricted to the training period only (so tuning itself can't peek at the future either), then a final honest evaluation on the untouched 2012-2023 period.
5. Saves a final tuned model as a *separate* artifact (`camel_tuned_model.joblib`) — this notebook does **not** overwrite the Phase 3 baseline (`camel_baseline_model.joblib`) that the backend currently serves; swapping the backend over is a deliberate follow-up decision, not an automatic side effect of running this notebook.

**How this connects onward**: same as Phase 3 — the chosen model is exactly what Phase 5's SHAP layer explains, so a compatibility check (`shap.TreeExplainer` on the chosen model) is included below before anything is saved.

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SRC_PATH = str(Path("..").resolve() / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from camel_sentinel.features import panel as panel_mod
from camel_sentinel.models import baseline, registry

print("Environment ready.")

Environment ready.


## 1. Build (or load cached) the multi-quarter panel

`panel.build_multi_quarter_panel()` fetches one quarter's financials at a time from the live FDIC API and stacks them — there is no synthetic multi-quarter fallback (unlike Phase 1-3's single-quarter `build_synthetic_sample()`), since a realistic *panel* of correlated failures across a crisis isn't something worth faking. This step needs real internet access to `banks.data.fdic.gov`.

Quarters chosen: **quarterly** through 2005-2013 (captures the pre-crisis build-up, the 2008-2012 crisis itself, and its immediate aftermath — where the overwhelming majority of real FDIC failures cluster), plus a handful of **annual** year-end snapshots through 2023 for post-crisis coverage. Fetching ~35 quarters takes a few minutes, so the result is cached to `data/processed/camel_panel.parquet` (gitignored, like every other pulled dataset in this project — reproducible from the API, not versioned) and reused on reruns.

In [2]:
PANEL_PATH = Path("..").resolve() / "data" / "processed" / "camel_panel.parquet"

if PANEL_PATH.exists():
    panel = pd.read_parquet(PANEL_PATH)
    print(f"Loaded cached panel from {PANEL_PATH}")
else:
    quarters = [f"{y}{q}" for y in range(2007, 2014) for q in ["0331", "0630", "0930", "1231"]]
    quarters += [f"{y}1231" for y in [2005, 2006, 2015, 2017, 2019, 2021, 2023]]
    panel = panel_mod.build_multi_quarter_panel(quarters, horizon_days=730)
    PANEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    panel.to_parquet(PANEL_PATH)
    print(f"Built panel and cached to {PANEL_PATH}")

print(f"Panel: {len(panel):,} bank-quarters | failed-within-2-years: {panel['failed'].sum():,} ({panel['failed'].mean():.3%})")

Loaded cached panel from C:\Users\raush\Downloads\AI-ML-Project-Afgan\Project\data\processed\camel_panel.parquet
Panel: 265,114 bank-quarters | failed-within-2-years: 4,165 (1.571%)


## 2. Prepare features/target

Same `baseline.prepare_training_data()` used in Phase 3 — the six CAMEL ratio columns, median-filled — but now pulling from the pooled panel instead of one quarter. `groups = panel["CERT"]` is kept alongside `X, y` so the cross-validation below can split *by bank*, not by row.

In [3]:
X, y = baseline.prepare_training_data(panel)
groups = panel["CERT"]

n_pos, n_neg = int(y.sum()), int((y == 0).sum())
print(f"X: {X.shape} | positive rate: {y.mean():.3%} | unique banks: {groups.nunique():,}")

X: (265114, 6) | positive rate: 1.571% | unique banks: 9,554


## 3. Candidate models

One representative per family: a linear baseline (logistic regression), the current Phase 3 model (random forest) plus a same-family variant (extra trees), three gradient-boosting implementations (sklearn's histogram-based booster, XGBoost, LightGBM — all handle the six ratios' non-linear interactions well and pair with `shap.TreeExplainer`), AdaBoost, a Naive Bayes baseline, and two imbalance-specific ensembles from `imbalanced-learn` (`BalancedRandomForest`, `EasyEnsemble` — both internally rebalance each tree/estimator's bootstrap sample instead of relying on `class_weight` alone).

`scale_pos_weight` (XGBoost) / `class_weight="balanced"` (everything else that supports it) re-weight the loss so the ~1.6% minority class isn't drowned out by "always predict healthy" — the same reasoning as Phase 3's baseline, just applied consistently across every model here.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV, cross_validate

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier

scale_pos_weight = n_neg / n_pos

def scaled(clf):
    # Logistic regression is the only candidate here that's scale-sensitive —
    # every tree/boosting model is invariant to monotonic per-feature scaling.
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

models = {
    "LogisticRegression": scaled(LogisticRegression(class_weight="balanced", max_iter=2000)),
    "RandomForest (Phase 3 baseline)": RandomForestClassifier(
        n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        random_state=42, verbosity=0, n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1,
        class_weight="balanced", random_state=42, verbosity=-1, n_jobs=-1,
    ),
    "GaussianNB": GaussianNB(),
    "BalancedRandomForest (imblearn)": BalancedRandomForestClassifier(
        n_estimators=300, max_depth=6, random_state=42, sampling_strategy="all", replacement=True, n_jobs=-1
    ),
    "EasyEnsemble (imblearn)": EasyEnsembleClassifier(n_estimators=20, random_state=42, n_jobs=-1),
}
print(f"{len(models)} candidate models.")

10 candidate models.


## 4. Grouped cross-validation comparison

`StratifiedGroupKFold(n_splits=5)` grouped by `CERT`: stratified so each fold keeps the ~1.6% positive rate, grouped so a bank's rows from adjacent quarters never split across train and test — without that, a model could partly "memorize" a specific bank's ratios rather than learn a general pattern, inflating every score.

Ranked by **PR-AUC** (average precision), not accuracy or even ROC-AUC — with a ~1.6% positive rate, PR-AUC is the metric that actually separates "found a real pattern" from "predicted the majority class." This cell takes several minutes (the panel has 265k rows).

In [5]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "balanced_accuracy": "balanced_accuracy",
}

rows = []
for name, model in models.items():
    t0 = time.time()
    result = cross_validate(model, X, y, cv=cv, groups=groups, scoring=scoring, n_jobs=1, error_score=np.nan)
    rows.append({
        "model": name,
        "ROC_AUC": np.nanmean(result["test_roc_auc"]),
        "PR_AUC": np.nanmean(result["test_avg_precision"]),
        "F1": np.nanmean(result["test_f1"]),
        "Recall": np.nanmean(result["test_recall"]),
        "Precision": np.nanmean(result["test_precision"]),
        "Balanced_Accuracy": np.nanmean(result["test_balanced_accuracy"]),
    })
    print(f"done: {name} ({time.time() - t0:.0f}s)")

results_df = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
results_df

done: LogisticRegression (9s)


done: RandomForest (Phase 3 baseline) (169s)


done: ExtraTrees (35s)


done: HistGradientBoosting (73s)


done: AdaBoost (162s)


...

done: LightGBM (17s)


done: GaussianNB (4s)


done: BalancedRandomForest (imblearn) (66s)


done: EasyEnsemble (imblearn) (84s)


,model,ROC_AUC,PR_AUC,F1,Recall,Precision,Balanced_Accuracy
3,HistGradientBoosting,0.9503,0.5144,0.2476,0.8491,0.1453,0.8846
6,LightGBM,0.9468,0.5103,0.2539,0.8320,0.1501,0.8784
5,XGBoost,0.9470,0.5080,0.2545,0.8321,0.1506,0.8785
9,EasyEnsemble (imblearn),0.9498,0.5045,0.2063,0.8632,0.1173,0.8798
8,BalancedRandomForest (imblearn),0.9521,0.4896,0.2468,0.8551,0.1444,0.8871
2,ExtraTrees,0.9412,0.4861,0.2560,0.8157,0.1522,0.8715
1,RandomForest (Phase 3 baseline),0.9511,0.4860,0.2653,0.8345,0.1579,0.8818
0,LogisticRegression,0.9423,0.4612,0.2399,0.8360,0.1404,0.8770
4,AdaBoost,0.9474,0.4522,0.4692,0.3828,0.6128,0.6894
7,GaussianNB,0.9255,0.3276,0.3963,0.3961,0.4046,0.6932


**Reading this table**: every model here separates failed from healthy banks meaningfully (ROC-AUC 0.92-0.95, versus 0.5 for the single-quarter comparison) — this is what "enough positive examples" looks like, unlike Phase 3's 15-example evaluation. Two genuinely different operating points show up:

- **Gradient-boosted trees and the imbalance-aware ensembles** (HistGradientBoosting, LightGBM, XGBoost, EasyEnsemble, BalancedRandomForest) catch ~83-86% of future failures at low precision (~0.12-0.16) — an early-warning screen, appropriate when missing a real failure is far costlier than a false alarm.
- **Sklearn's `GradientBoosting`/`AdaBoost`-style models** trade the other way: lower recall, much higher precision — a short, high-confidence watchlist instead of a broad screen.

`HistGradientBoostingClassifier` tops PR-AUC and trains in seconds (native histogram binning, no external dependency beyond scikit-learn itself) — the model this notebook carries forward into the stricter checks below.

**But this grouped CV still isn't the blueprint's "cross-validated on time (no look-ahead)" ask** — folds mix quarters from every year together, so a fold's test set can still contain 2008-2009-crisis-era rows the model effectively trained on very similar rows for (same regime, different bank). Section 5 below is the real test: train only on 2005-2011, evaluate only on a future the model never saw at all.

## 5. No-look-ahead time-based validation

Split the panel **by calendar quarter**, not randomly: train on 2005-2011 (the run-up to and through the worst of the crisis — 3,802 of the panel's 4,165 positives), evaluate only on 2012-2023 (363 positives) — a period with a visibly different failure regime (the tail of the crisis winding down, then a long, much calmer stretch). A model that only learned "2008-2009 was bad" and nothing generalizable should fall apart here; a model that learned something about *what makes a bank's ratios look distressed* should still work, just at a lower positive rate.

Comparing the Phase 3 baseline model family (RandomForest) against the untuned Phase 4 winner (HistGradientBoosting), both refit on the training period only, both scored on the same untouched future period — a fair, apples-to-apples check before any tuning happens.

In [6]:
TIME_SPLIT = "20120101"
train_mask = panel["REPDTE"] < TIME_SPLIT
test_mask = ~train_mask

X_train_time, y_train_time = X[train_mask], y[train_mask]
X_test_time, y_test_time = X[test_mask], y[test_mask]

print(f"Train (2005-2011): {len(X_train_time):,} rows, {y_train_time.sum():,} failed ({y_train_time.mean():.3%})")
print(f"Test  (2012-2023): {len(X_test_time):,} rows, {y_test_time.sum():,} failed ({y_test_time.mean():.3%})")

Train (2005-2011): 181,169 rows, 3,802 failed (2.099%)
Test  (2012-2023): 83,945 rows, 363 failed (0.432%)


In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score, balanced_accuracy_score

def evaluate_on_future(model, name):
    model.fit(X_train_time, y_train_time)
    proba = model.predict_proba(X_test_time)[:, 1]
    pred = model.predict(X_test_time)
    return {
        "model": name,
        "ROC_AUC": roc_auc_score(y_test_time, proba),
        "PR_AUC": average_precision_score(y_test_time, proba),
        "F1": f1_score(y_test_time, pred),
        "Recall": recall_score(y_test_time, pred),
        "Precision": precision_score(y_test_time, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_test_time, pred),
    }

time_rows = [
    evaluate_on_future(
        RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1),
        "RandomForest (untuned, Phase 3 style)",
    ),
    evaluate_on_future(
        HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
        "HistGradientBoosting (untuned)",
    ),
]
time_results_df = pd.DataFrame(time_rows)
time_results_df

,model,ROC_AUC,PR_AUC,F1,Recall,Precision,Balanced_Accuracy
0,"RandomForest (untuned, Phase 3 style)",0.9634,0.4949,0.1841,0.9008,0.1025,0.9333
1,HistGradientBoosting (untuned),0.9692,0.5475,0.1735,0.9174,0.0958,0.9399


## 6. Hyperparameter tuning — restricted to the training period only

`RandomizedSearchCV` over `HistGradientBoostingClassifier`'s main knobs, scored on **PR-AUC** (`average_precision` — the imbalance-appropriate metric, same reasoning as section 4), cross-validated with `StratifiedGroupKFold` grouped by bank — but critically, run **only on the 2005-2011 training rows**. The 2012-2023 period stays completely untouched by tuning, so the evaluation in the next cell is an honest answer to "how would this generalize to data it never influenced at all," not just "did we overfit the CV folds." 

In [8]:
from scipy.stats import randint, uniform

param_distributions = {
    "max_iter": randint(100, 500),
    "learning_rate": uniform(0.02, 0.28),          # 0.02-0.30
    "max_leaf_nodes": randint(15, 63),
    "min_samples_leaf": randint(20, 200),
    "l2_regularization": uniform(0.0, 2.0),
    "max_depth": [None, 3, 4, 6, 8],
}

search = RandomizedSearchCV(
    estimator=HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
    param_distributions=param_distributions,
    n_iter=25,
    scoring="average_precision",
    cv=StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    refit=True,
)

t0 = time.time()
search.fit(X_train_time, y_train_time, groups=groups[train_mask])
print(f"RandomizedSearchCV done in {time.time() - t0:.0f}s")
print("Best CV PR-AUC (train period only):", round(search.best_score_, 4))
print("Best params:", search.best_params_)

RandomizedSearchCV done in 71s
Best CV PR-AUC (train period only): 0.5262
Best params: {'l2_regularization': 0.5925470114081648, 'learning_rate': 0.0662747429376407, 'max_depth': None, 'max_iter': 426, 'max_leaf_nodes': 23, 'min_samples_leaf': 107}


## 7. Final honest comparison — all three models, scored on the untouched 2012-2023 future

Same evaluation as section 5, now adding the tuned model — `search.best_estimator_` was already refit on the full 2005-2011 training set by `RandomizedSearchCV(refit=True)`, so this is a direct apples-to-apples three-way comparison on data none of the three ever trained on.

In [9]:
tuned_time_row = evaluate_on_future(search.best_estimator_, "HistGradientBoosting (tuned)")
final_time_results_df = pd.concat([time_results_df, pd.DataFrame([tuned_time_row])], ignore_index=True)
final_time_results_df = final_time_results_df.sort_values("PR_AUC", ascending=False)
final_time_results_df

,model,ROC_AUC,PR_AUC,F1,Recall,Precision,Balanced_Accuracy
2,HistGradientBoosting (tuned),0.9727,0.5489,0.1712,0.9201,0.0944,0.9409
1,HistGradientBoosting (untuned),0.9692,0.5475,0.1735,0.9174,0.0958,0.9399
0,"RandomForest (untuned, Phase 3 style)",0.9634,0.4949,0.1841,0.9008,0.1025,0.9333


**Reading this table**: this is the strictest, most honest evaluation in the project so far — three models, all trained only on 2005-2011 data, all scored on a 2012-2023 period none of them (or their hyperparameters) ever saw. Whatever gap exists between the tuned and untuned HistGradientBoosting rows here is a real improvement from tuning, not an artifact of the CV folds being too easy; whatever gap exists between HistGradientBoosting and RandomForest here is a real generalization difference, not a same-regime coincidence from section 4's grouped-but-not-time-based CV.

In [10]:
FINAL_PARAMS = search.best_params_
final_model = HistGradientBoostingClassifier(class_weight="balanced", random_state=42, **FINAL_PARAMS)
final_model.fit(X, y)  # refit on the FULL panel (train+test period) for deployment, now that
                        # hyperparameters were chosen using only the training period above
print(f"Refit tuned HistGradientBoosting on the full panel ({len(X):,} rows) for deployment.")

Refit tuned HistGradientBoosting on the full panel (265,114 rows) for deployment.


## 8. SHAP compatibility check

The entire reason Phase 3 picked a tree ensemble (see `models/baseline.py`'s module docstring) is that `shap.TreeExplainer` is exact and fast for tree ensembles, versus the slower, approximate model-agnostic explainers a non-tree model would need. Confirming that guarantee still holds for the tuned model, on a small sample, before saving it as a candidate replacement.

In [11]:
import shap

explainer = shap.TreeExplainer(final_model)
sample = X.sample(5, random_state=42)
shap_values = explainer.shap_values(sample)
print("TreeExplainer output shape:", np.shape(shap_values), "— exact SHAP values computed successfully.")

TreeExplainer output shape: (5, 6) — exact SHAP values computed successfully.


## 9. Save as a separate, non-production artifact

`registry.save_model(model, filename=...)` — saved under a *different* filename than Phase 3's `camel_baseline_model.joblib`, so this notebook never silently changes what the running backend serves. Pointing `backend/main.py` at `camel_tuned_model.joblib` instead is a deliberate follow-up step, not something this notebook does on its own.

In [12]:
saved_path = registry.save_model(final_model, filename="camel_tuned_model.joblib")
print(f"Saved to {saved_path}")

results_df.to_csv(Path("model_comparison_results_panel.csv"), index=False)
final_time_results_df.to_csv(Path("model_comparison_results_time_holdout.csv"), index=False)
print("Saved comparison tables to notebooks/model_comparison_results_panel.csv and model_comparison_results_time_holdout.csv")

Saved to C:\Users\raush\Downloads\AI-ML-Project-Afgan\Project\models\camel_tuned_model.joblib
Saved comparison tables to notebooks/model_comparison_results_panel.csv and model_comparison_results_time_holdout.csv


## 10. What's next

Phase 4 is now done to the blueprint's actual spec: a real multi-quarter panel (265k bank-quarters, 4,165 positives), a broad model-family comparison, a genuine no-look-ahead time-based validation (train 2005-2011, test 2012-2023), and hyperparameter tuning restricted to the training period only. Still ahead:

- **Backend swap**: point `backend/main.py` at `camel_tuned_model.joblib` once satisfied with the tradeoff in section 7 (and pick the operating point — broad early-warning screen vs. high-confidence watchlist — deliberately, since that's a policy choice, not just a metric to maximize).
- **Threshold tuning**: everything above used each model's default 0.5 threshold; the recall/precision tradeoff can be tuned further via `predict_proba` + a custom cutoff, informed by section 7's precision/recall numbers.
- **Phase 3 gaps**: peer clustering (KMeans/HDBSCAN) and anomaly detection (Isolation Forest) from the blueprint's Phase 3 row are still unbuilt.
- **Phase 6 (RAI)**: the fairness audit across bank-size/region/charter-type still needs to run against whichever model is actually deployed.